# 01 — Data Initialization

Audit and reuse the frozen five-class BDD100K data for Round 2. Check original annotations, source hashes, native labels, geometry and role/group disjointness without resplitting, relabeling or rewriting historical evidence.

- **Inputs:** Existing #14 five-class pairs, original BDD sources, frozen protocol/subsets, historical six-class conversions and v2 policy.
- **Outputs:** `docs/round2-data-v1/`: immutable audit, compressed source/role/group membership, support and sampled visual-quality evidence.
- **Prerequisites:** Project study kernel and all saved artifacts in section 1; this notebook never prepares missing inputs.

**Outline**

1. [Configuration and Preflight](#configuration-preflight)
2. [Source and Five-Class Integrity Audit](#source-audit)
3. [Class, Size and Visual Quality Evidence](#quality-evidence)
4. [Frozen Membership and Audit Summary](#audit-summary)

Set paths, then **Restart & Run All**. Saved prerequisites are required; upstream work is never launched. The original split-building notebook is preserved byte-for-byte under `archive/round1/`.

<a id="configuration-preflight"></a>

## 1. Configuration and Preflight

In [ ]:
import gzip
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from matplotlib.patches import Rectangle

from vehicle_detection.common import (
    digest,
    file_hash,
    freeze_json,
    read_image,
    read_json,
    require_paths,
)
from vehicle_detection.evaluation_data import audit_data, validate_audit
from vehicle_detection.five_class import FIVE_CLASSES, remap_boxes
from vehicle_detection.protocol import validate_protocol
from vehicle_detection.stress import diagnostics

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
DATA_ROOT = ROOT / "data/bdd100k"
PROTOCOL_PATH = ROOT / "runs/issue5-v1/protocol.json"
SUBSETS_PATH = ROOT / "runs/issue6-v2/subsets.json"
POLICY_PATH = ROOT / "docs/issue6-v2/policy.json"
FIVE_ROOT = ROOT / "runs/preprocessing-five-v1"
TRAIN_CONVERSION = ROOT / "runs/issue6-v2/conversion/S10"
DEV_CONVERSION = ROOT / "runs/issue5-v1/conversion"
OUTPUT = ROOT / "docs/round2-data-v1"
QUALITY_SAMPLE_PER_ROLE = 16
SEED = 2026
require_paths(
    data=DATA_ROOT,
    protocol=PROTOCOL_PATH,
    subsets=SUBSETS_PATH,
    policy=POLICY_PATH,
    five_class=FIVE_ROOT / "summary.json",
    training_labels=TRAIN_CONVERSION / "conversion.json",
    development_labels=DEV_CONVERSION / "conversion.json",
)
for protected in (
    DATA_ROOT,
    FIVE_ROOT,
    PROTOCOL_PATH.parent,
    SUBSETS_PATH.parent,
    POLICY_PATH.parent,
    TRAIN_CONVERSION,
    DEV_CONVERSION,
):
    a, b = OUTPUT.resolve(), protected.resolve()
    if a == b or a in b.parents or b in a.parents:
        raise ValueError(f"Output overlaps preserved data: {protected}")
if OUTPUT.exists() and any(OUTPUT.iterdir()) and not (OUTPUT / "audit.json").is_file():
    raise ValueError("Unknown report directory; choose a new output root")
if type(QUALITY_SAMPLE_PER_ROLE) is not int or QUALITY_SAMPLE_PER_ROLE < 1:
    raise ValueError("Quality sample size must be a positive integer")
cv2.setNumThreads(1)
plt.rcParams.update({"figure.dpi": 90, "font.size": 9})
protocol = validate_protocol(read_json(PROTOCOL_PATH))
print("Reuse frozen protocol:", protocol["manifest_id"])

<a id="source-audit"></a>

## 2. Source and Five-Class Integrity Audit

All non-final source images are hashed and their headers checked. Both existing Raw/Adaptive S10 and development pairs receive deep label/hash/geometry checks in read-only mode. Official-val images receive path-existence checks only; final support below comes from saved annotations. Missing or stale inputs raise errors; no repair, resplit or label conversion runs.

In [ ]:
audit = audit_data(
    protocol,
    DATA_ROOT,
    FIVE_ROOT,
    read_json(SUBSETS_PATH),
    {"training": TRAIN_CONVERSION, "development": DEV_CONVERSION},
    read_json(POLICY_PATH),
)
for path in (PROTOCOL_PATH, SUBSETS_PATH, POLICY_PATH):
    audit["artifacts"][str(path)] = file_hash(path)
if (OUTPUT / "audit.json").exists():
    previous = validate_audit(read_json(OUTPUT / "audit.json"), protocol)
    if (
        any(previous.get(k) != v for k, v in audit.items() if k not in {"audit_id", "artifacts"})
        or any(previous["artifacts"].get(k) != v for k, v in audit["artifacts"].items())
        or previous["quality_sample"]["seed"] != SEED
        or previous["quality_sample"]["per_role_limit"] != QUALITY_SAMPLE_PER_ROLE
    ):
        raise ValueError("Audit inputs/sample settings changed; choose a new output root")
support = pd.DataFrame(audit["role_support_annotation_only"]).fillna(0).astype(int)
keys = [
    "images",
    "new_negative_images",
    "newly_negative_images",
    *[f"new_instances:{name}" for name in FIVE_CLASSES],
]
display(support.reindex(keys, fill_value=0).T)
display(pd.DataFrame(audit["exports"]).T)
print("Source discovery:", audit["discovery"])
print("Original box exclusions/errors:", protocol["label_audit"])
print("Group safeguards:", protocol["group_audit"])
print("\n".join(protocol["limitations"]))

<a id="quality-evidence"></a>

## 3. Class, Size and Visual Quality Evidence

In [ ]:
quality_rows, examples = [], {}
for role, rows in protocol["splits"].items():
    if role == "final_test":
        continue
    sample = sorted(rows, key=lambda r: (digest([SEED, r["id"]]), r["id"]))[
        :QUALITY_SAMPLE_PER_ROLE
    ]
    for row in sample:
        image = read_image(DATA_ROOT / row["source_path"])
        quality_rows.append({"role": role, "id": row["id"], **diagnostics(image)})
        examples.setdefault(role, (row, image))
quality = pd.DataFrame(quality_rows)
OUTPUT.mkdir(parents=True, exist_ok=True)
quality.to_csv(OUTPUT / "quality_sample.csv", index=False)
display(
    quality.groupby("role")[["brightness", "contrast", "sharpness", "noise_highpass"]].agg(
        ["count", "median", "min", "max"]
    )
)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
support.reindex([f"new_instances:{c}" for c in FIVE_CLASSES], fill_value=0).T.plot.bar(
    ax=axes[0],
    title="Five-class box counts (final: annotations only)",
    rot=20,
)
support.reindex([f"new_size:{s}" for s in ("small", "medium", "large")], fill_value=0).T.plot.bar(
    ax=axes[1],
    title="Native-pixel object sizes",
    rot=20,
)
fig.tight_layout()
fig.savefig(OUTPUT / "role_support.png")
plt.show()
fig, axes = plt.subplots(1, len(examples), figsize=(15, 4), squeeze=False)
for ax, (role, (row, image)) in zip(axes[0], examples.items(), strict=True):
    ax.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    for cls, x1, y1, x2, y2 in remap_boxes(row["boxes"]):
        ax.add_patch(
            Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor="lime", linewidth=0.6)
        )
        ax.text(x1, y1, FIVE_CLASSES[cls], fontsize=5, color="yellow")
    ax.set_title(f"{role}\n{row['id']}")
    ax.axis("off")
fig.tight_layout()
fig.savefig(OUTPUT / "source_examples.png")
plt.show()
print("Quality describes a fixed sample of original train-role pixels; no policy or AP evidence.")

<a id="audit-summary"></a>

## 4. Frozen Membership and Audit Summary

In [ ]:
membership = pd.DataFrame(
    [
        {"role": role, **{k: row[k] for k in ("id", "group", "source_group", "source_path")}}
        for role, rows in protocol["splits"].items()
        for row in rows
    ]
).sort_values(["role", "id"])
content = gzip.compress(membership.to_csv(index=False).encode(), mtime=0)
path = OUTPUT / "source_role_membership.csv.gz"
if path.exists() and path.read_bytes() != content:
    raise ValueError("Frozen membership differs; choose a new output root")
if not path.exists():
    path.write_bytes(content)
audit["quality_sample"] = {
    "seed": SEED,
    "per_role_limit": QUALITY_SAMPLE_PER_ROLE,
    "ids": quality.groupby("role").id.apply(list).to_dict(),
}
audit["artifacts"][str(path)] = file_hash(path)
audit["artifacts"][str(OUTPUT / "quality_sample.csv")] = file_hash(OUTPUT / "quality_sample.csv")
audit["audit_id"] = digest({k: v for k, v in audit.items() if k != "audit_id"})
freeze_json(OUTPUT / "audit.json", audit)
print("PASS: reused sources/roles/labels and read-only deep pair verification")
print("Saved prerequisite for notebook 02:", OUTPUT / "audit.json")
print("Final holdout stays sealed; no training, inference or historical rewrites.")